# Agentic AI Systems
## by <i>Andreas Grotz</i>

## Table of Contents
- [Introduction](#intro)
- [Imports and Initializations](#import)
- [Research Agent](#research)
- [Review Agent](#review)
- [Agentic Workflow](#flow)
- [Evaluation of the Workflow](#eval)
- [Summary](#summary)

<a id='intro'></a>
### Introduction

In this project, we will design, implement, and evaluate a small-scale agentic AI system using agent design and orchestration techniques from Udacity's Agentic AI Nanodegree. Our use case is a web-based competitor analysis in marketing research. We define a research agent with limited memory and access to a web search tool that produces the report, as well as a stateless review agent that checks the report for specific criteria and provides feedback. The workflow is then set up as a research-review loop that runs until the review agent accepts the report.

We start by some imports and initializations. Then we define the research and the review agent, tie them together in an agentic workflow, and evaluate that workflow for different scenarios. A short summary concludes the notebook.

Note that in order to re-run the notebook, you need to have valid API keys for OpenAI and Tavily, which you need to include in the ".env" file contained in the repository.


<a id='import'></a>
### Imports and Initializations

In this section, we import required Python modules, and we initialize the OpenAI model and the Tavily client.

In [1]:
#!pip install dotenv
#!pip install pydantic
#!pip install pydantic_ai
#!pip install tavily

In [2]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIResponsesModel, OpenAIResponsesModelSettings 
from pydantic_ai.providers.openai import OpenAIProvider
from tavily import TavilyClient

In [3]:
# Load environment variables
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
DEFAULT_OPENAI_MODEL="gpt-4o-mini"
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
if OPENAI_API_KEY[0:3] == "voc":
    BASE_URL = "https://openai.vocareum.com/v1"
else :
    BASE_URL = None

In [4]:
# Initialize clients
provider=OpenAIProvider(api_key=OPENAI_API_KEY,base_url = BASE_URL)
openai_model = OpenAIResponsesModel(DEFAULT_OPENAI_MODEL, provider=provider)
model_settings = OpenAIResponsesModelSettings(temperature=0.0) 
# Zero temperature for more consistency, but note that this does not ensure 100% reproducibility
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

<a id='research'></a>
### Research Agent

In this section, we define the research agent producing the competitor report. We start by extending pydantic's Agent class to include memory in form of the chat history.

In [5]:
class AgentWithMemory(Agent):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.reset_memory()

    def reset_memory(self):
        self.message_history = []

    async def run(self, prompt):
        result = await super().run(prompt, message_history=self.message_history)
        self.message_history += result.new_messages()
        return result

Next, we define the system prompt for the research agent:

In [6]:
research_system_prompt = """
ROLE

You are an experienced marketing researcher, specialized in web-based competitor analysis.

TASK

1) Your task is to perform a competitor analysis for a company and a business area provided in the user prompt.
2) You are responsible for planning your research task, developing meaningful search queries, and synthesize the results of your analysis in a report.
3) For research planning and query development, you can use your internal knowledge, 
   but for identifying and analyzing competitors you must rely on the web_search_tool and the chat history.
4) Your report should include 
    a) an introduction, 
    b) a short profile for each competitor with an explanation of their relevant offerings, 
    c) a short summary and recommendation.
5) All relevant statements on competitors in the report must be documented with sources.
6) In addition, you will document every thinking step of your research process in a research diary. 
   Explain the planning, the search queries, and the approach for the synthesis.
7) You may get feedback on your report that you then need to incorporate and update your report accordingly.
   You also need to amend the research diary with the steps you took to incorporate the feedback.
8) Regarding output, the report needs to be contained in the competitor_report field of the output format and the research diary in the research_diary field.
   You should not mix up the outputs!

"""


Now we define a pydantic model for the research agent's output scheme and we initialize the agent.

In [7]:
class ResearchResult(BaseModel):

    competitor_report: str = Field(description="The final competitor analysis report.")
    research_diary: str = Field(description="The research diary containing documentation of the research process.")

In [8]:
research_agent = AgentWithMemory(
    system_prompt=research_system_prompt,
    output_type=ResearchResult,
    model=openai_model,
    model_settings=model_settings
)

We finally define the web search tool and link it to the research agent.

In [9]:
@research_agent.tool_plain
def web_search_tool(query: str) -> list:
    """
    Semantic search: Search the web using Tavily API
    args:
    - query (str): Search query. 

    You'll receive results as list. Each element contains:
    - url: The web URL of the result
    - title: The title of the web page
    - content: The content of the web page
    - score: The relevance score of the search result
    """
    print(f"Running query: {query}")
    search_result = tavily_client.search(
        query=query,
        include_answer=True
    )
    print("Done searching")
    return search_result.get("results", "")

Let us test the research agent with a first business idea:

In [10]:
research_user_prompt = 'I want to open a coffee shop in San Francisco, where guests can play with their dogs while drinking their beverage.'

In [11]:
research_result = await research_agent.run(research_user_prompt)

Running query: dog-friendly coffee shops in San Francisco
Done searching
Running query: coffee shop competitors San Francisco dog-friendly
Running query: pet-friendly cafes San Francisco
Running query: dog-friendly businesses San Francisco
Done searching
Done searching
Done searching
Running query: San Francisco coffee shop market analysis
Done searching
Running query: San Francisco coffee shop trends 2023
Done searching


As we can see from the above log, the agent triggered several web searches via the web_search_tool. Let us look at look at the report, which seems to meet our requirements:

In [12]:
print(research_result.output.competitor_report)

# Competitor Analysis Report: Dog-Friendly Coffee Shops in San Francisco

## Introduction
San Francisco is renowned for its vibrant coffee culture and dog-friendly atmosphere. This report analyzes the competitive landscape for a new coffee shop that caters to dog owners, providing a space where guests can enjoy their beverages while their dogs play. The analysis includes profiles of existing competitors, their offerings, and insights into market trends.

## Competitor Profiles

### 1. **Spike's Coffee and Teas**  
- **Location:** Various locations in San Francisco  
- **Offerings:** Known for its wide variety of coffee and tea options, Spike's is a popular spot for dog owners due to its outdoor seating that welcomes pets.  
- **Source:** [Rover Blog](https://www.rover.com/blog/dog-friendly-coffee-shops-san-francisco/)

### 2. **Duboce Park Cafe**  
- **Location:** Near Duboce Park  
- **Offerings:** This cafe is famous for its relaxed atmosphere and proximity to a dog park, making it a

We also required the agent to document its process in a research diary. This shows genuine planning, reasoning and decision efforts:

In [13]:
print(research_result.output.research_diary)

1. **Planning:** Initiated research to identify competitors in the dog-friendly coffee shop market in San Francisco, focusing on their offerings and market trends.
2. **Search Queries:** Utilized various search queries including "dog-friendly coffee shops in San Francisco," "pet-friendly cafes San Francisco," and "San Francisco coffee shop market analysis" to gather comprehensive data.
3. **Data Collection:** Collected information from multiple sources, including blogs, Yelp, and market analysis reports, to ensure a well-rounded understanding of the competitive landscape.
4. **Synthesis:** Analyzed the data to create detailed competitor profiles and identify key market trends, ensuring all statements were backed by credible sources.
5. **Report Compilation:** Compiled findings into a structured report, highlighting competitor offerings, market trends, and strategic recommendations for entering the market.


We conclude the section by checking that the agent's memory is filled as expected:

In [14]:
research_agent.message_history[0]

ModelRequest(parts=[SystemPromptPart(content='\nROLE\n\nYou are an experienced marketing researcher, specialized in web-based competitor analysis.\n\nTASK\n\n1) Your task is to perform a competitor analysis for a company and a business area provided in the user prompt.\n2) You are responsible for planning your research task, developing meaningful search queries, and synthesize the results of your analysis in a report.\n3) For research planning and query development, you can use your internal knowledge, \n   but for identifying and analyzing competitors you must rely on the web_search_tool and the chat history.\n4) Your report should include \n    a) an introduction, \n    b) a short profile for each competitor with an explanation of their relevant offerings, \n    c) a short summary and recommendation.\n5) All relevant statements on competitors in the report must be documented with sources.\n6) In addition, you will document every thinking step of your research process in a research di

<a id='review'></a>
### Review Agent

In this section, we define the review agent that checks the report for specific criteria. We start by crafting a sensible system prompt and defining a pydantic model for the review agent's output scheme.

In [15]:
review_system_prompt = """
ROLE

You are a reviewer for marketing reports. Your review process follows clear rules specified below.

TASK

1) You will review competitor analysis reports produced by a marketing researcher. 
2) In the user prompt, you receive both the report request and the final report.
3) Your task is to review the report and to decide if you accept or reject the report.
4) Your review and decision will be based EXCLUSIVELY on the following requirements:
        a) Does the report contain a short profile of each competitor?
        b) Does the report provide at least one web resource per competitor?
        c) Does the report include a summary or recommendation?
        d) Is the report specific to the request?
5) You MUST NOT consider additional or more detailed requirements for your review!
6) Think in steps! For example, first identify all mentioned competitors, then check requirements a) and b) for each competitors separately.
7) You will provide an overall decision whether you accept the report, as well as detailed feedback with respect to each requirement.
8) If you reject the report, then your feedback needs to be very specific and actionable, so the researcher can improve the report based on your feedback.
9) In addition, you will provide a detailed review diary, where you document each step of your review.

"""

In [16]:
class ReviewResult(BaseModel):

    decision: bool = Field(description="True if all evaluation criteria are met and the report is accepted, otherwise False.")
    feedback: str = Field(description="A detailed feedback on each requirement.")
    review_diary: str = Field(description="The research diary containing documentation of the research process.")

Now we initialize the agent. Note that unlike the research agent, which requires memory to improve on its previously generated reports, the review agent is stateless as it only needs to review the latest iteration of the report.

In [17]:
review_agent = Agent(
    system_prompt=review_system_prompt,
    output_type=ReviewResult,
    model=openai_model,
    model_settings=model_settings
)

Let us check the agent's verdict on the first test report regarding the dog cafe idea:

In [18]:
def get_review_prompt(user_prompt, report):
    review_prompt = f'''
    REPORT REQUEST: {user_prompt}
    
    FINAL REPORT: {report}
    '''
    return review_prompt

In [19]:
review_result = await review_agent.run(get_review_prompt(research_user_prompt, research_result.output.competitor_report))

As we can see, the agent accepted the report. The review diary, which is part of the required output, shows genuine reasoning and decision processes to come up with that conclusion:

In [20]:
print(review_result)

AgentRunResult(output=ReviewResult(decision=True, feedback='The report meets all the requirements: it includes a short profile of each competitor, provides at least one web resource per competitor, includes a summary and recommendations, and is specific to the request for a dog-friendly coffee shop in San Francisco.', review_diary='1. **Planning:** The research began with identifying the target market for a dog-friendly coffee shop in San Francisco. I aimed to understand existing competitors and market trends.\n2. **Search Queries:** I developed queries such as "dog-friendly coffee shops in San Francisco," "pet-friendly cafes San Francisco," and "San Francisco coffee shop market analysis" to gather relevant information.\n3. **Synthesis:** I analyzed the gathered data to create competitor profiles and identify market trends, ensuring to document sources for credibility. The final report was structured to provide a clear overview of the competitive landscape and actionable recommendation

<a id='flow'></a>
### Agentic Workflow

In this section, we tie the research and review agent together in a research-review loop that stops when either the review agent accepts the report or a maximum number of iterations is reached.

In [21]:
async def run_workflow(research_user_prompt, max_steps):
    research_agent.reset_memory() # clear memory for the new task
    research_result = await research_agent.run(research_user_prompt)
    review_result = await review_agent.run(get_review_prompt(research_user_prompt, research_result.output.competitor_report))
    print(review_result)
    print()
    if (review_result.output.decision):
        return (1, research_result, review_result)
    for step in range(1, max_steps):
        research_feedback_prompt = f'Please improve your report with regard to the following feedback: {review_result.output.feedback}'
        research_result = await research_agent.run(research_feedback_prompt)
        review_result = await review_agent.run(get_review_prompt(research_user_prompt, research_result.output.competitor_report))
        print(review_result)
        print()
        if (review_result.output.decision):
            return (step+1, research_result, review_result)
    return (max_steps, research_result, review_result)

Let us run the loop with our dog cafe scenario. As expected from the above sections and our zero temperature setting, the report is accepted after the first iteration and produces a report and review very similar to the above.

In [22]:
n_steps, research_result, review_result = await run_workflow(research_user_prompt, 5)

Running query: dog-friendly coffee shops San Francisco
Done searching
Running query: dog-friendly cafes San Francisco
Done searching
Running query: dog-friendly coffee shops San Francisco reviews
Done searching
Running query: dog-friendly coffee shops San Francisco features
Done searching
Running query: dog-friendly coffee shops San Francisco unique features
Done searching
AgentRunResult(output=ReviewResult(decision=True, feedback='The report meets all the requirements: it includes a short profile for each competitor, provides at least one web resource per competitor, includes a summary and recommendations, and is specific to the request for dog-friendly coffee shops in San Francisco.', review_diary='1. **Planning**: The research began with identifying the target market for a dog-friendly coffee shop in San Francisco. The focus was on existing competitors and their offerings.\n2. **Search Queries**: I used queries like "dog-friendly coffee shops San Francisco" and "dog-friendly cafes S

In [23]:
print(research_result.output.competitor_report)

# Competitor Analysis Report: Dog-Friendly Coffee Shops in San Francisco

## Introduction
San Francisco is a vibrant city known for its coffee culture and love for dogs. This analysis explores the competitive landscape of dog-friendly coffee shops in the area, focusing on their offerings, unique features, and customer experiences. The goal is to identify key competitors and understand their strengths and weaknesses to inform the launch of a new dog-friendly coffee shop.

## Competitor Profiles

### 1. **HITW Coffee**  
- **Overview**: Known for its relaxed atmosphere and friendly service, HITW Coffee is a popular spot among dog owners.  
- **Offerings**: Exceptional coffee, light snacks, and a welcoming environment for pets.  
- **Unique Features**: Outdoor seating that accommodates dogs, making it a favorite for pet owners.  
- **Source**: [City Dogs San Francisco](https://citydogsanfrancisco.com/blog/dog-friendly-coffee-shops-san-francisco/)

### 2. **Java Beach Cafe**  
- **Overview

<a id='eval'></a>
### Evaluation of the Workflow
In this section, we look at some more scenarios. Let us start with another creative idea in the food space:

In [24]:
research_user_prompt_breakfast = 'We are a restaurant chain specialized in Full English Breakfast, and we want to open a franchise in Tokyo.'
n_steps_breakfast, research_result_breakfast, eval_result_breakfast = await run_workflow(research_user_prompt_breakfast, 5)

Running query: Full English Breakfast restaurants in Tokyo
Done searching
Running query: Full English Breakfast competitors TokyoRunning query: English breakfast restaurants Tokyo reviews

Done searching
Done searching
Running query: Full English Breakfast Tokyo restaurant reviews competitors
Done searching
Running query: Full English Breakfast Tokyo restaurant competitors analysis
Done searching
Running query: Full English Breakfast Tokyo restaurant competitors overview
Done searching
Running query: Full English Breakfast Tokyo restaurant competitors summary
Done searching
AgentRunResult(output=ReviewResult(decision=True, feedback='The report meets all the specified requirements. It includes a short profile of each competitor, provides at least one web resource per competitor, includes a summary and recommendations, and is specific to the request for Full English Breakfast restaurants in Tokyo.', review_diary="1. Reviewed the report request to identify the focus on Full English Breakf

In [25]:
print(research_result_breakfast.output.competitor_report)

# Competitor Analysis Report: Full English Breakfast Restaurants in Tokyo

## Introduction
As a restaurant chain specializing in Full English Breakfast, entering the Tokyo market presents both opportunities and challenges. This report analyzes the current landscape of Full English Breakfast offerings in Tokyo, identifying key competitors and their unique selling propositions.

## Competitor Profiles
1. **HP and Yorkshire Tea**  
   - **Overview**: This establishment is known for its authentic Full English Breakfast, featuring eggs, bacon, sausages, mushrooms, tomatoes, beans, toast, and hash browns.  
   - **Unique Selling Proposition**: They emphasize quality ingredients, including Heinz beans, and a traditional British breakfast experience.  
   - **Source**: [Tokyo Cheapo](https://tokyocheapo.com/food-and-drink/international-food/tokyo-english-breakfast/)

2. **Nutty's Caff**  
   - **Overview**: A popular spot among locals and expats, Nutty's Caff offers a hearty Full English Break

Next, we consider a more sophisticated request from the consulting industry:

In [26]:
research_user_prompt_consulting = '''We are a European consulting company, specialized in digital services that comply with EU data protection requirements. 
In your analysis, please identify and include the top three consulting firms in the market, as well as any other competitors you deem relevant.'''
n_steps_consulting, research_result_consulting, eval_result_consulting = await run_workflow(research_user_prompt_consulting, 5)

Running query: top consulting firms digital services EU data protection compliance 2023
Done searching
Running query: top consulting firms GDPR compliance Europe 2023
Running query: leading data protection consulting firms Europe 2023
Done searching
Done searching
Running query: top consulting firms data protection compliance Europe 2023
Done searching
Running query: top consulting firms data protection compliance Europe 2023
Done searching
Running query: top consulting firms data protection compliance Europe 2023
Done searching
Running query: top consulting firms GDPR compliance Europe 2023
Done searching
Running query: top consulting firms GDPR compliance Europe 2023
Done searching
AgentRunResult(output=ReviewResult(decision=True, feedback='The report meets all the specified requirements. It includes a short profile for each competitor, provides at least one web resource per competitor, includes a summary and recommendations, and is specific to the request for consulting firms specia

In [27]:
print(research_result_consulting.output.competitor_report)

### Competitor Analysis Report: Digital Consulting Firms in Europe Specializing in GDPR Compliance

#### Introduction
As a European consulting company specializing in digital services that comply with EU data protection requirements, understanding the competitive landscape is crucial. This report identifies the top consulting firms in the market, focusing on their offerings related to GDPR compliance and data protection services.

#### Competitor Profiles

1. **Accenture**  
   - **Overview**: A global consultancy with extensive experience in GDPR compliance across various sectors.  
   - **Offerings**: Accenture provides end-to-end privacy solutions, including compliance assessments, data protection officer services, and training programs to help organizations navigate GDPR requirements.  
   - **Source**: [LinkedIn](https://www.linkedin.com/pulse/top-gdpr-consulting-service-companies-how-compare-them-zxz1f/)

2. **Deloitte**  
   - **Overview**: A leading consulting firm known for it

So far, all reports were accepted in the first iteration. Let us see if we can come up with something that needs at least a second iteration by considering the following ambitious idea in the area of physics and engineering:

In [28]:
research_user_prompt_altfacts = '''
I have a bullet-proof idea for a conventional perpetuum mobile that revolutionizes physics and can supply energy to the world forever.
'''
n_steps_altfacts, research_result_altfacts, eval_result_altfacts = await run_workflow(research_user_prompt_altfacts, 5)

AgentRunResult(output=ReviewResult(decision=False, feedback='1. **Competitor Profiles**: The report contains a short profile for each competitor, which meets the requirement.  \n2. **Web Resources**: Each competitor profile includes a web resource, fulfilling this requirement as well.  \n3. **Summary or Recommendation**: The report includes a summary and recommendations, which is satisfactory.  \n4. **Specificity to Request**: The report does not specifically address the request for a competitor analysis related to a conventional perpetuum mobile. Instead, it focuses on renewable energy companies, which is not aligned with the original request.  \n\nTo improve the report, ensure that the analysis is directly related to the concept of a perpetual motion machine or similar innovations, rather than general renewable energy solutions.', review_diary='1. **Planning**: The goal was to analyze competitors in the renewable energy sector, focusing on their offerings and innovations.  \n2. **Sea

In [29]:
print(research_result_altfacts.output.competitor_report)

### Competitor Analysis Report on Perpetual Motion Machines

#### Introduction
The concept of perpetual motion machines has intrigued inventors and scientists for centuries, despite being deemed impossible by the laws of thermodynamics. This report analyzes recent innovations and attempts in the field of perpetual motion machines, focusing on their designs, claims, and the scientific community's response to these endeavors.

#### Competitor Profiles

1. **WO/2023/063908 Perpetual Motion Machine**  
   - **Overview**: This patent describes a mechanical machine designed to achieve permanent and self-rotating movement. The periodic movement is claimed to be renewed with the same strength as its initial activity.  
   - **Relevant Offerings**: The design proposes a mechanism that could theoretically operate indefinitely without external energy input, challenging conventional physics.  
   - **Source**: [WIPO Patent WO/2023/063908](https://patentscope.wipo.int/search/en/WO2023063908)

2. **

In [30]:
print(research_result_altfacts.output.research_diary)

1. **Planning**: The goal was to analyze competitors specifically related to perpetual motion machines, focusing on recent innovations and patents.
2. **Search Queries**: I developed queries such as "recent perpetual motion machine innovations 2023" and "perpetual motion machine patents 2023" to gather relevant information.
3. **Synthesis Approach**: I compiled profiles of competitors based on their claims, designs, and the scientific community's response. Each profile includes a brief overview and relevant offerings, supported by credible sources to ensure accuracy and reliability. I ensured that the analysis directly addressed the request for a competitor analysis related to perpetual motion machines, rather than general renewable energy solutions.


As we can see, the research agent flags the phyiscal impossibility of a perpetuum mobile and instead comes up with competitors in the area of renewable energy solutions, which is rejected by the review agent. In a second iteration, the research agent then focuses on direct competitors for pertpetuum mobiles (as documented in its research diary), and comes up with some interesting companies, which is accepted by the review agent.

<a id='summary'></a>
### Summary

We have implemented an agentic workflow for competitor analysis in marketing, which consists of a research agent and a review agent interacting in cycles. The research agent has memory and access to a web search tool, while the review agent is stateless. The underlying LLM is OpenAI's <i>gpt-4o-mini</i> model. 

Both agents behave largely as intended:
- The research agent uses the web search tool, crafts well-written reports with citations for several scenarios (dog cafe in San Francisco, breakfast restaurant in Tokyo, consulting services for digital services in Europe) and documents its planning and reasoning process in a research diary.
- The review agent reviews and accepts these reports according to criteria, documenting its reasoning and decisionmaking in a review diary.
- In the challenging case of a perpetuum mobile, a second iteration is required as the research agent focussed instead on renewable energy solution in the first iteration. It focusses on the specific request in the second iteration, while still flagging the counterfactual nature of such an endeavor.

A limitation of the current setup is that the review agent only checks on a formal level whether sources are provided for the statements made in the report. An improved version would for example provide the review agent with a tool to request and retrieve the content of the web pages mentioned in the report, so that it can check whether the statements in the report are actually in line with the cited sources.

In [31]:
# Let us finally generate the requirements file for reproducibility.
!pip freeze > requirements.txt